In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# 1. Load the synthetic data you generated earlier
df = pd.read_csv('/kaggle/input/datasets/witherstorm/pricing-data/task_pricing_data.csv')

# 2. Define the Rule-Based Formula
def calculate_rule_based(row):
    base_price = 0
    if row['Category'] == 'Cleaning': base_price = 50
    elif row['Category'] == 'Delivery': base_price = 30
    elif row['Category'] == 'Plumbing': base_price = 80
    elif row['Category'] == 'Moving Help': base_price = 100
    
    urgency_fee = 20 if row['Is_Urgent'] == 1 else 0
    
    # Formula: Base + ($2 * km) + ($10 * hours) + Urgency Fee
    return base_price + (row['Distance_km'] * 2) + (row['Duration_hours'] * 10) + urgency_fee

# Apply the rule to create our baseline predictions
df['Rule_Based_Price'] = df.apply(calculate_rule_based, axis=1)

print("Data loaded and Rule-Based prices calculated.")
df.head()

Data loaded and Rule-Based prices calculated.


,Category,Distance_km,Duration_hours,Is_Urgent,Price,Rule_Based_Price
0,Moving Help,14.6,4.5,1,192.88,194.2
1,Moving Help,8.1,3.1,0,146.92,147.2
2,Plumbing,11.1,2.5,1,150.05,147.2
3,Moving Help,10.6,1.1,0,129.16,132.2
4,Plumbing,18.2,3.0,1,164.59,166.4


In [3]:
# Select the features the AI will use to predict the price
X = df[['Category', 'Distance_km', 'Duration_hours', 'Is_Urgent']]

# Convert text categories into binary columns
X_encoded = pd.get_dummies(X, columns=['Category'], drop_first=True)

# Define the target variable (the actual price with random noise)
y = df['Price']

# Split the data: 80% for training the AI, 20% for testing it
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print("Data preprocessed and split into training/testing sets.")

Data preprocessed and split into training/testing sets.


In [4]:
# Train the Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions on the test set
lr_predictions = lr_model.predict(X_test)

print("AI Model Trained!")

AI Model Trained!


In [5]:
# Get the Rule-Based predictions just for the test set
rule_based_test_predictions = df.loc[X_test.index, 'Rule_Based_Price']

# Calculate Mean Absolute Error (How many dollars off are we on average?)
rule_mae = mean_absolute_error(y_test, rule_based_test_predictions)
ai_mae = mean_absolute_error(y_test, lr_predictions)

print("--- STATISTICAL COMPARISON ---")
print(f"Rule-Based Average Error: ${rule_mae:.2f}")
print(f"AI Model Average Error:   ${ai_mae:.2f}")

# Calculate Confidence Band (Margin of Error)
# We use the standard deviation of the AI's errors to create a price range
errors = y_test - lr_predictions
std_dev_error = np.std(errors)

print(f"\nAI Price Band (Confidence Range): +/- ${std_dev_error:.2f}")
print("Example: If AI predicts $50, we show the user a range of ${:.2f} to ${:.2f}".format(
    50 - std_dev_error, 50 + std_dev_error
))

--- STATISTICAL COMPARISON ---
Rule-Based Average Error: $2.31
AI Model Average Error:   $2.36

AI Price Band (Confidence Range): +/- $2.72
Example: If AI predicts $50, we show the user a range of $47.28 to $52.72


In [6]:
# Save the trained model
joblib.dump(lr_model, 'pricing_regressor.pkl')

# Save the exact column names (Crucial for the API later to match formatting)
joblib.dump(list(X_encoded.columns), 'model_columns.pkl')

print("Model and columns saved successfully to the /models directory!")

Model and columns saved successfully to the /models directory!
